# Spectral Decomposition Approach


Simplifed noteebook demonstrating night sky decomposition approach.

Sky spectrum decomposed into model of airglow sky lines and continuum components:

I. Continuum components include:

- Moon (high-resolution solar spectrum rebinned to LVM sampling and convolved to LVM Gaussian LSF and multiplied by B-spline multiplicative continuum). This mimicks Moon itself and Zodi components.

- Diffuse components (interpolated from low-resolution PALACE diffuse continuum components)

  - Hydroperoxyl ($HO_2$): This is the predominant continuum component in the near-infrared range, characterized by a prominent emission peak at 1.51 µm.

  - Iron Monoxide ($FeO$) and other molecules: This component dominates the visual wavelength range (roughly 500 to 720 nm) and includes the $FeO$ "orange arc" bands, with potential additional contributions from $NiO$ or $OFeOH$.

  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).


II. Airglow sky components include:

  - Atomic Oxygen (O I) emission lines within the visual wavelength range.

  - Sodium (Na I): doublet at 5889.95 and 5895.92 Å, formed in the mesospheric Na layer at about 92 km by chemiluminescent reactions of meteoric sodium; typical D2 / D1 ≈ 1.7.
  
  - Potassium (K I): doublet at 7664.90 and 7698.96 Å, formed in the mesospheric K layer at about 89 km by chemistry similar to Na; typical D2 / D1 ≈ 1.67.
  
  - Nitrogen (N I): [ N I ] [NI] doublet at 5197.90 and 5200.26 Å, formed higher in the ionosphere at about 250 km via dissociative recombination; typical 5198 / 5200 ≈ 1.76.


  - Unresolved Molecular Oxygen ($O_2$): Located in the ultraviolet (UVB) range, this component accounts for weak, unresolved bands from high-energy electronic states (specifically $c^1\Sigma^-_u$, $A'^3\Delta_u$, and $A^3\Sigma^+_u$).

  - Hydroxyl (OH): This component accounts for the hydroxyl emission lines in the visual wavelength range (roughly 500 to 720 nm).

# Manual in-place experiments

## Imports and helpers

Load the core packages and define the helper functions used throughout the notebook.

In [1]:
from pathlib import Path
import os

import clarabel
import numpy as np
import plotly.graph_objects as go
import scipy.sparse as sp
from astropy.io import fits
from astropy.table import Table
from scipy.interpolate import BSpline
from scipy.optimize import minimize, minimize_scalar
from scipy.stats import norm as sp_norm, exponnorm
from lvmdrp.core.fluxcal import rebin_and_convolve


def vac_to_air(lam_vac_a):
    lam = np.asarray(lam_vac_a, float)
    s2 = (1e4 / lam) ** 2
    n = 1.0 + 8.34254e-5 + 2.406147e-2 / (130.0 - s2) + 1.5998e-4 / (38.9 - s2)
    return lam / n


def decode_HITRAN_ID(table):

    ids = table['ID'].astype(str)

    # Slice the string to get different components by position
    # Format: OHXX [4]v' [5]v'' [6]dN [7]dJ [8]F' [9]F'' [10:12]N'' [12]parity
    ids_str = np.asarray(ids, dtype=str)

    # Guard against unexpected ID formats (prevents confusing int conversion errors).
    if np.min(np.char.str_len(ids_str)) < 13:
        raise ValueError("HITRAN ID strings are shorter than expected (need positions up to 12).")

    v_up_str = np.array([s[4:5] for s in ids_str])
    v_low = np.array([s[5:6] for s in ids_str], dtype=int)
    branch_n = np.array([s[6:7] for s in ids_str])
    branch_j = np.array([s[7:8] for s in ids_str])
    f_up = np.array([s[8:9] for s in ids_str], dtype=int)
    f_low = np.array([s[9:10] for s in ids_str], dtype=int)
    n_low = np.array([s[10:12] for s in ids_str], dtype=int)
    parity = np.array([s[12:13] for s in ids_str])

    # In PMD notation, the symbol 'X' for v_upper means level 10
    v_up_str = np.where(v_up_str == 'X', '10', v_up_str)
    v_up = v_up_str.astype(int)

    # Calculate the starting value N_upper using the branch shift
    delta_map = {'O': -2, 'P': -1, 'Q': 0, 'R': 1, 'S': 2}
    d_n = np.vectorize(delta_map.get)(branch_n)
    n_up = n_low + d_n

    # Add clear columns to the input table
    table['v_upper'] = v_up
    table['v_lower'] = v_low
    table['N_upper'] = n_up
    table['N_lower'] = n_low
    table['F_upper'] = f_up
    table['F_lower'] = f_low
    table['branch_N'] = branch_n
    table['branch_J'] = branch_j
    table['parity'] = parity

    # Add a flag: Main line (True) or Satellite (False)
    # (Main: spin does not change, branches are P, Q, R)
    table['is_main'] = (f_up == f_low) & np.isin(branch_n, ['P', 'Q', 'R'])

    return table


def grp2vector(line_wave, line_amp, wave, lsf):
    cent = np.asarray(line_wave, float)
    amp = np.asarray(line_amp, float)
    sig = np.interp(cent, wave, lsf) if np.ndim(lsf) > 0 else float(lsf)

    yy = (wave[:, None] - cent) / sig

    return np.sum(amp[None, :] * np.exp(-0.5 * yy**2), axis=1)

FACTOR = 1e14
LSF_SIGMA = 0.5
T_O2 = 191.5 # in K


In [2]:
from pathlib import Path
import os
from astropy.io import fits
f = Path(os.environ["SAS_BASE_DIR"]) / "sdsswork/lvm/spectro/redux/amjones/XSFrame_1.2.1.fits"
fits.info(f)


wave = fits.getdata(f, "WAVE").astype(np.float64)
flx_skye = fits.getdata(f, "SKYE_FLUX").astype(np.float64) * FACTOR
flx_skyw = fits.getdata(f, "SKYW_FLUX").astype(np.float64) * FACTOR
flx_flux = fits.getdata(f, "FLUX").astype(np.float64) * FACTOR
flx_sky = fits.getdata(f, "SKY").astype(np.float64) * FACTOR
flx_ivar = fits.getdata(f, "IVAR").astype(np.float64) / FACTOR**2
flx_err = 1.0 / np.sqrt(flx_ivar)
flx_sci = flx_flux + flx_sky

Filename: /Users/ik52/obs/sas/sdsswork/lvm/spectro/redux/amjones/XSFrame_1.2.1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       5   ()      
  1  WAVE          1 ImageHDU         7   (12401,)   float32   
  2  FLUX          1 ImageHDU        20   (12401, 16625)   float32   
  3  SKY           1 ImageHDU        20   (12401, 16625)   float32   
  4  IVAR          1 ImageHDU        20   (12401, 16625)   float32   
  5  SKYE_FLUX     1 ImageHDU        20   (12401, 16625)   float32   
  6  SKYW_FLUX     1 ImageHDU        20   (12401, 16625)   float32   
  7  FLUX_P20      1 ImageHDU        20   (12401, 16625)   float64   
  8  SKY_P20       1 ImageHDU        20   (12401, 16625)   float64   
  9  SKYE_P20      1 ImageHDU        20   (12401, 16625)   float64   
 10  SKYW_P20      1 ImageHDU        20   (12401, 16625)   float64   
 11  DRP_ALL       1 BinTableHDU    141   16625R x 66C   [6A, K, K, K, D, K, K, K, 5A, K, 59A, 23A, D, D, D, 

## Prepare matrix with OH line groups

Build grouped OH emission templates on the observed wavelength grid using the local LSF.

In [3]:
foh = "palace/PMD/pmd_popmodel_OH.dat"

oh = Table.read(foh, format="ascii.basic", guess=False, comment="#", fast_reader=False)
oh["wave"] = vac_to_air(np.asarray(oh["lam"], float) * 1e4)

cap = 5.0
mask_oh = (oh["wave"] >= wave.min() - cap) & (oh["wave"] <= wave.max() + cap)
oh = oh[mask_oh]
oh = decode_HITRAN_ID(oh)
oh["band"] = np.char.add(
    np.asarray(oh["v_upper"], str),
    np.char.add("-", np.asarray(oh["v_lower"], str)),
)
oh["branch"] = np.char.add(
    np.asarray(oh["branch_N"], str),
    np.asarray(oh["branch_J"], str),
)

OH_GROUP_KEYS = ("v_upper", "N_upper", "F_upper")

groups = oh.group_by(OH_GROUP_KEYS).groups

matrix_OH = np.zeros((len(groups), wave.size))

for igrp, grp in enumerate(groups):
    amp = grp['Aij'] * grp['gi']
    matrix_OH[igrp] = grp2vector(grp["wave"], amp, wave, LSF_SIGMA)

matrix_OH.shape

(402, 12401)

## Prepare moon components as solar spectrum derivatives

Construct the Moon block as a rebinned solar spectrum multiplied by a B-spline basis.

In [4]:
sol = np.loadtxt("Spectre_HR_LATMOS_Meftah_V1_350_1000nm.txt", comments=";")
solar_wave = vac_to_air(sol[:, 0] * 10.0)
solar_flux = np.asarray(sol[:, 1], float)
solar_flux /= np.nanmedian(solar_flux)

lsf_sigma = 1.5

solar_rb = rebin_and_convolve(
    wave, solar_wave, solar_flux,
    LSF_SIGMA * 2.355, lsf_in_wavelength=True
)
solar_rb /= np.nanmedian(solar_rb)

spline_degree = 3
N_SPLINE_KNOTS = 25

w0, w1 = wave[0], wave[-1]
interior = np.linspace(w0, w1, N_SPLINE_KNOTS + 2)[1:-1]
t_knots = np.r_[(w0,) * (spline_degree + 1), interior, (w1,) * (spline_degree + 1)]

matrix_Bspl = BSpline.design_matrix(wave, t_knots, spline_degree).toarray()
matrix_Moon = (solar_rb[:, None] * matrix_Bspl).T

moon_names = [f"Moon_bs{i:02d}" for i in range(matrix_Moon.shape[0])]
matrix_Moon.shape

(29, 12401)

## Create matrix with diffuse PALACE continuum components

Interpolate the low-resolution PALACE diffuse continuum components onto the spectral grid without LSF convolution.

In [5]:
fref = "palace/PMD/pmd_refcont.dat"

ref = Table.read(fref, format="ascii")
lam_ref = np.asarray(ref["lam"], float) * 1e4

vector_HO2 = np.interp(wave, lam_ref, np.asarray(ref["fcHO2"], float))
vector_FeO = np.interp(wave, lam_ref, np.asarray(ref["fcFeO"], float))
vector_O2Ac = np.interp(wave, lam_ref, np.asarray(ref["fcO2Ac"], float))

vector_HO2 = np.nan_to_num(vector_HO2, nan=0.0, posinf=0.0, neginf=0.0)
vector_FeO = np.nan_to_num(vector_FeO, nan=0.0, posinf=0.0, neginf=0.0)
vector_O2Ac = np.nan_to_num(vector_O2Ac, nan=0.0, posinf=0.0, neginf=0.0)

matrix_DIFFUSE = np.vstack([
    vector_HO2,
    vector_FeO,
    vector_O2Ac,
])

diffuse_names = ["HO2", "FeO", "O2Ac"]
matrix_DIFFUSE.shape, diffuse_names

((3, 12401), ['HO2', 'FeO', 'O2Ac'])

## Create matrix with atomic sky lines

Build atomic airglow templates from PALACE and render them on the observed grid.

### OI lines
All lines tied with OI0845 or OI0777

In [6]:
forc = "palace/PMD/pmd_intmodel_Orc.dat"

orc = Table.read(forc, format="ascii.basic", guess=False, comment="#", fast_reader=False)
orc["wave"] = vac_to_air(np.asarray(orc["lam"], float) * 1e4)

mask_orc = (orc["wave"] >= wave.min() - cap) & (orc["wave"] <= wave.max() + cap)
orc = orc[mask_orc]

groups_orc = orc.group_by("reffeat").groups

matrix_Orc = np.zeros((len(groups_orc), wave.size))
orc_names = []

for igrp, grp in enumerate(groups_orc):
    matrix_Orc[igrp] = grp2vector(grp["wave"], grp["I"], wave, LSF_SIGMA)
    orc_names.append(f"ATOM_Orc_{grp['reffeat'][0]}")

matrix_Orc.shape, orc_names

((2, 12401), ['ATOM_Orc_OI0777', 'ATOM_Orc_OI0845'])

### Other atomic lines

In [7]:
fatom = "palace/PMD/pmd_intdata_atom.dat"

atom = Table.read(fatom, format="ascii.basic", guess=False, comment="#", fast_reader=False)
atom["wave"] = vac_to_air(np.asarray(atom["lam"], float) * 1e4)

mask_atom = (atom["wave"] >= wave.min() - cap) & (atom["wave"] <= wave.max() + cap)
atom = atom[mask_atom]
# exclude H and OI lines
atom = atom[~np.isin(np.asarray(atom["class"], str), ["H", "Orc"])]

groups_atom = atom.group_by("class").groups

matrix_ATOM = np.zeros((len(groups_atom), wave.size))
atom_names = []

for igrp, grp in enumerate(groups_atom):
    amp = np.asarray(grp["I"], float)
    amp /= np.nansum(amp)
    matrix_ATOM[igrp] = grp2vector(grp["wave"], amp, wave, LSF_SIGMA)
    atom_names.append(f"ATOM_{grp['class'][0]}")

matrix_ATOM.shape, atom_names

((5, 12401), ['ATOM_K', 'ATOM_N', 'ATOM_Na', 'ATOM_Og', 'ATOM_Or'])

### O2 molecular band

For fixed temperature $T_{O2}$ defined in the first code cell of this notebook.

In [8]:
fo2 = "palace/PMD/pmd_popmodel_O2.dat"
o2_min, o2_max = 8600.0, 8715.0
hc_over_kB_cmK = 1.4387769

pop_o2 = Table.read(fo2, format="ascii.basic", guess=False, comment="#", fast_reader=False)
pop_o2["wave"] = vac_to_air(np.asarray(pop_o2["lam"], float) * 1e4)

o2 = pop_o2[
    (pop_o2["wave"] >= o2_min) &
    (pop_o2["wave"] <= o2_max) &
    (np.asarray(pop_o2["vi"], int) == 0)
]

lam_o2 = np.asarray(o2["wave"], float)
Ei = np.asarray(o2["Ei"], float)
Aij = np.asarray(o2["Aij"], float)
gi = np.asarray(o2["gi"], float)
E0 = float(np.nanmin(Ei))

rel_o2 = Aij * gi * np.exp(-hc_over_kB_cmK * (Ei - E0) / T_O2)
rel_o2 /= np.nansum(rel_o2)

matrix_O2 = grp2vector(lam_o2, rel_o2, wave, LSF_SIGMA)[np.newaxis, :]
o2_names = ["O2_b01"]

matrix_O2.shape, o2_names

((1, 12401), ['O2_b01'])

## Solve the problem

Assemble the full design matrix, solve the non-negative QP with Clarabel, and compute fit diagnostics.

In [9]:
design_matrix = np.vstack([
    matrix_OH,
    matrix_Moon,
    matrix_DIFFUSE,
    matrix_ATOM,
    matrix_Orc,
    matrix_O2,
])

design_names = (
    [f"OH_{i:03d}" for i in range(matrix_OH.shape[0])] +
    moon_names +
    diffuse_names +
    atom_names +
    orc_names +
    o2_names
)

idx = 78
msk_good = np.isfinite(flx_sci[idx]) & np.isfinite(flx_ivar[idx])
y = flx_sci[idx, msk_good]
w = np.sqrt(flx_ivar[idx, msk_good])

n_good = int(np.sum(msk_good))

A = design_matrix[:, msk_good].T
Aw = A * w[:, None]
yw = y * w

P_dense = Aw.T @ Aw
q = -(Aw.T @ yw)
n_par = int(P_dense.shape[0])

P = sp.csc_matrix((P_dense + P_dense.T) / 2.0)
P = sp.triu(P).tocsc()
q = np.asarray(q, dtype=np.float64)

A_con = -sp.eye(n_par, format="csc")
b_con = np.zeros(n_par, dtype=np.float64)
cones = [clarabel.NonnegativeConeT(n_par)]

settings = clarabel.DefaultSettings()
settings.verbose = False

import time
t_fit0 = time.perf_counter()
solver = clarabel.DefaultSolver(P, q, A_con, b_con, cones, settings)
qp_result = solver.solve()
fit_elapsed_sec = time.perf_counter() - t_fit0
coef = np.asarray(qp_result.x, float)

bestfit = design_matrix.T @ coef
resid = flx_sci[idx] - bestfit
rms = np.nanstd(resid)
resid_level = -3.0 * rms

i0 = 0
i1 = i0 + matrix_OH.shape[0]
i2 = i1 + matrix_Moon.shape[0]
i3 = i2 + matrix_DIFFUSE.shape[0]
i4 = i3 + matrix_ATOM.shape[0]
i5 = i4 + matrix_Orc.shape[0]
i6 = i5 + matrix_O2.shape[0]

comp_OH = matrix_OH.T @ coef[i0:i1]
comp_Moon = matrix_Moon.T @ coef[i1:i2]
comp_HO2 = coef[i2] * vector_HO2
comp_FeO = coef[i2 + 1] * vector_FeO
comp_O2Ac = coef[i2 + 2] * vector_O2Ac
comp_DIFFUSE = comp_HO2 + comp_FeO + comp_O2Ac
comp_ATOM = matrix_ATOM.T @ coef[i3:i4]
comp_Orc = matrix_Orc.T @ coef[i4:i5]
comp_O2 = matrix_O2.T @ coef[i5:i6]

chi2 = float(np.sum(resid[msk_good]**2 * flx_ivar[idx, msk_good]))
dof = max(n_good - n_par, 1)
reduced_chi2 = chi2 / dof

fit_summary = (
    f"status={qp_result.status} | npar={n_par} | ngood={n_good} | "
    f"chi2_red={reduced_chi2:.4f} | dt={fit_elapsed_sec:.2f}s"
)

print(fit_summary)
coef.shape, qp_result.status, reduced_chi2, fit_elapsed_sec

status=Solved | npar=442 | ngood=12401 | chi2_red=25751.5690 | dt=0.29s


((442,), Solved, 25751.56904060763, 0.28856283298227936)

In [10]:
err_plot = np.where(flx_ivar[idx] > 0, 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], 1.0)), np.nan)

err_plot = 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], np.nan))


fig = go.Figure()
fig.add_trace(go.Scattergl(x=wave, y=flx_sci[idx], mode="lines", name="Observed",
                            line=dict(color="black", width=1)))
fig.add_trace(go.Scattergl(x=wave, y=bestfit, mode="lines", name="Best-fit full model",
                            line=dict(color="crimson", width=1.5)))
fig.add_trace(go.Scattergl(x=wave, y=comp_Moon, mode="lines", name="Moon",
                            line=dict(color="#4e79a7", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=comp_DIFFUSE, mode="lines", name="Diffuse",
                            line=dict(color="#59a14f", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=comp_ATOM, mode="lines", name="Atomic",
                            line=dict(color="#f28e2b", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=comp_Orc, mode="lines", name="OI recomb",
                            line=dict(color="#b07aa1", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=comp_O2, mode="lines", name="O2",
                            line=dict(color="#76b7b2", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=resid_level + resid, mode="lines", name="Residual",
                            line=dict(color="royalblue", width=1)))
fig.add_trace(go.Scattergl(x=wave, y=resid_level + err_plot, mode="lines", name="+1sigma",
                            line=dict(color="gray", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=resid_level - err_plot, mode="lines", name="-1sigma",
                            line=dict(color="gray", width=1, dash="dash")))

fig.update_layout(
    title={
        'text': (
            f"XSFrame_1.2.1.fits idx={idx} | "
            f"T_O2={T_O2:.1f} K <br> {fit_summary}"
        ),
        'font': {'size': 14}
    },
    xaxis_title="λ (Å)", yaxis_title="Flux", template="plotly_white", height=520)
fig.show()

# Using Python module

In [11]:
%load_ext autoreload
%autoreload 2

In [55]:
from sky_decomp.fit import SkyDecomp

decomposer = SkyDecomp(wave, lsf_sigma=LSF_SIGMA)

In [62]:
idx = 50

result = decomposer.fit(
    flx_sci[idx],
    flx_ivar[idx],
    verbose=True,
    n_lsf_refits=3,
)

O2
  T         196.17 +/- 5.90 K
  chi2_red  4.195e+04
  dt        0.194 s

decomp
  init      3.015e+04

iterations
  [1] LSF     B=9756   R=2.038e+04   Z=1.689e+05   dt=0.004 s
      decomp  chi2_red=1.997e+04   qp=0.324 s
  [2] LSF     B=1741   R=6163   Z=3.861e+04   dt=0.004 s
      decomp  chi2_red=1.699e+04   qp=0.296 s
  [3] LSF     B=1747   R=5829   Z=3.741e+04   dt=0.003 s
      decomp  chi2_red=1.687e+04   qp=0.345 s

final
  decomp    1.687e+04
  refits    3
  total_dt  6.286 s
  peak_mem  251.86 MB


In [63]:
bestfit = result.bestfit
bestfit_lsf = result.bestfit_lsf
comp_Moon = result.components["moon"]
comp_DIFFUSE = result.components["diffuse"]

resid = flx_sci[idx] - bestfit
resid_lsf = flx_sci[idx] - bestfit_lsf
resid_level = -3.0 * np.nanstd(resid)

err_plot = 1.0 / np.sqrt(np.where(flx_ivar[idx] > 0, flx_ivar[idx], np.nan))

fig = go.Figure()
fig.add_trace(go.Scattergl(x=wave, y=flx_sci[idx], mode="lines", name="Observed",
                           line=dict(color="black", width=1)))
fig.add_trace(go.Scattergl(x=wave, y=bestfit_lsf, mode="lines", name="Best-fit + LSF",
                           line=dict(color="darkorange", width=1.5)))
fig.add_trace(go.Scattergl(x=wave, y=comp_Moon, mode="lines", name="Moon",
                           line=dict(color="#4e79a7", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=comp_DIFFUSE, mode="lines", name="Diffuse",
                           line=dict(color="#59a14f", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=resid_level + resid_lsf, mode="lines", name="Residual + LSF",
                           line=dict(color="seagreen", width=1)))
fig.add_trace(go.Scattergl(x=wave, y=resid_level + err_plot, mode="lines", name="+1sigma",
                           line=dict(color="gray", width=1, dash="dash")))
fig.add_trace(go.Scattergl(x=wave, y=resid_level - err_plot, mode="lines", name="-1sigma",
                           line=dict(color="gray", width=1, dash="dash")))

fig.update_layout(
    title={
        "text": (
            f"idx={idx} | T_O2={result.t_o2:.1f}±{result.t_o2_err:.1f} K<br>"
            f"{result.fit_summary}"
        ),
        "font": {"size": 14},
    },
    xaxis_title="λ (Å)",
    yaxis_title="Flux",
    template="plotly_white",
    height=520,
)
fig.show()


In [64]:
from scipy.optimize import curve_fit
from scipy.special import erf
from plotly.subplots import make_subplots

koff = np.arange(-5, 6)
xe = np.arange(-5.5, 6.0, 1.0)
xf = np.linspace(-5.5, 5.5, 1200)
chs = {"B": "#4e79a7", "R": "#e15759", "Z": "#59a14f"}

def g(x, a, mu, s):
    return a * np.exp(-0.5 * ((x - mu) / s) ** 2)

def gb(x, a, mu, s):
    u0 = (x - 0.5 - mu) / (np.sqrt(2) * s)
    u1 = (x + 0.5 - mu) / (np.sqrt(2) * s)
    return a * np.sqrt(np.pi / 2) * s * (erf(u1) - erf(u0))

def step(y):
    return np.repeat(xe, 2)[1:-1], np.repeat(y, 2)

pars, titles = {}, []
for ch in chs:
    ker = result.lsf_kernels.get(ch)
    try:
        p, _ = curve_fit(gb, koff, ker, p0=(1, 0, 1), bounds=([0, -2, 0.2], [10, 2, 5]))
        pars[ch] = tuple(map(float, p))
        titles.append(f"{ch}: μ={p[1]:.2f}, σ={p[2]:.2f} px")
    except Exception:
        titles.append(f"{ch}: fit failed")

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=titles)

for i, (ch, color) in enumerate(chs.items(), start=1):
    ker = result.lsf_kernels.get(ch)
    if ker is None:
        continue

    for ref, c in chs.items():
        if ref not in pars:
            continue
        fig.add_scatter(
            x=xf, y=g(xf, *pars[ref]), mode="lines", name=f"{ref} Gaussian",
            line=dict(color=c, width=1.4),
            opacity=1.0 if ref == ch else 0.28,
            showlegend=(i == 1), row=1, col=i,
        )

    xs, ys = step(ker)
    fig.add_scatter(
        x=xs, y=ys, mode="lines", name=f"{ch} LSF",
        line=dict(color=color, width=2.4),
        showlegend=(i == 1), row=1, col=i,
    )

    if ch in pars:
        xs, ys = step(gb(koff, *pars[ch]))
        fig.add_scatter(
            x=xs, y=ys, mode="lines", name=f"{ch} binned fit",
            line=dict(color="black", width=1.5, dash="dash"),
            showlegend=(i == 1), row=1, col=i,
        )

fig.update_layout(
    title=dict(text="Recovered channel LSF kernels", x=0.5, y=0.97, xanchor="center"),
    template="plotly_white",
    height=380,
    margin=dict(l=60, r=30, t=85, b=55),
    legend=dict(x=1.02, y=1.0, xanchor="left", yanchor="top"),
)
fig.update_xaxes(title_text="Kernel pixel offset", dtick=1)
fig.update_yaxes(title_text="Kernel value", row=1, col=1)
fig.show()

result.lsf_metrics


{'B': {'status': 'Solved',
  'reason': '',
  'n_pixels': 4374,
  'n_valid_pixels': 4374,
  'valid_frac': 1.0,
  'chi2': 7622137.756839683,
  'chi2_red': 1746.9946726655244,
  'r2': 0.9482469497803199,
  'rms_resid': 0.47223308351222903,
  'runtime_sec': 0.0009764169808477163,
  'sum_kernel': 0.9999999999999999,
  'center_pix': -0.02189401591312192,
  'sigma_pix': 1.4218741966061017},
 'R': {'status': 'Solved',
  'reason': '',
  'n_pixels': 3334,
  'n_valid_pixels': 3334,
  'valid_frac': 1.0,
  'chi2': 19371071.68331794,
  'chi2_red': 5829.392622123966,
  'r2': 0.8348422743860006,
  'rms_resid': 0.5278575312776521,
  'runtime_sec': 0.0009580421028658748,
  'sum_kernel': 1.0000000000000002,
  'center_pix': -0.0025004259690113227,
  'sigma_pix': 1.0526768433514644},
 'Z': {'status': 'Solved',
  'reason': '',
  'n_pixels': 4693,
  'n_valid_pixels': 4692,
  'valid_frac': 0.9997869166844237,
  'chi2': 175094185.45841601,
  'chi2_red': 37405.29490673275,
  'r2': 0.8888870093742101,
  'rms_res